# Assignment Beta — Question 4: Evaluating an LLM Sentiment Classifier
**Course:** Agentic AI (AI4015), FAST NUCES Karachi  **Student ID:** 22K-8706

300 customer reviews, 3 classes (Positive / Negative / Neutral). Stage 1 = zero-shot prompt, Stage 2 = prompt with an
explicit Neutral definition + 6 few-shot examples. All metrics are implemented from first principles and cross-checked
against `nltk` / `rouge_score` where available.

In [1]:
import math
from collections import Counter
import numpy as np

CLASSES = ["Positive", "Negative", "Neutral"]
CM1 = np.array([[100, 5, 15],
                [8, 80, 12],
                [20, 10, 50]])          # Stage 1: zero-shot   (rows = actual, cols = predicted)
CM2 = np.array([[106, 4, 10],
                [6, 84, 10],
                [9, 7, 64]])            # Stage 2: definition + 6 few-shot examples
Z = 1.959964                            # 97.5th percentile of N(0,1)

def accuracy_ci(cm):
    n = cm.sum(); k = np.trace(cm); p = k / n
    se = math.sqrt(p * (1 - p) / n)
    wald = (p - Z * se, p + Z * se)
    centre = (p + Z**2 / (2 * n)) / (1 + Z**2 / n)
    half = Z * math.sqrt(p * (1 - p) / n + Z**2 / (4 * n**2)) / (1 + Z**2 / n)
    return dict(n=int(n), correct=int(k), acc=p, se=se, wald=wald, wilson=(centre - half, centre + half))

def per_class(cm):
    rows = []
    for i, c in enumerate(CLASSES):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        p = tp / (tp + fp); r = tp / (tp + fn); f = 2 * p * r / (p + r)
        rows.append(dict(cls=c, tp=int(tp), fp=int(fp), fn=int(fn), support=int(cm[i].sum()), P=p, R=r, F1=f))
    return rows

def averages(cm):
    pc = per_class(cm); sup = np.array([r["support"] for r in pc])
    tp = sum(r["tp"] for r in pc); fp = sum(r["fp"] for r in pc); fn = sum(r["fn"] for r in pc)
    micro_p = tp / (tp + fp); micro_r = tp / (tp + fn)
    return dict(
        macro_P=np.mean([r["P"] for r in pc]), macro_R=np.mean([r["R"] for r in pc]),
        macro_F1=np.mean([r["F1"] for r in pc]),
        weighted_F1=float(np.sum(sup * np.array([r["F1"] for r in pc])) / sup.sum()),
        micro_P=micro_p, micro_R=micro_r, micro_F1=2 * micro_p * micro_r / (micro_p + micro_r))

# ---- prompt stability --------------------------------------------------------
RUNS = np.array([0.847, 0.840, 0.853, 0.837, 0.850])

# ---- generated justification ---------------------------------------------------
REF = "the delivery was late but the product quality is good".split()
CAND = "the product quality is good but delivery was slow".split()
TOKEN_PROBS = [0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20]

def ngrams(toks, n):
    return Counter(tuple(toks[i:i + n]) for i in range(len(toks) - n + 1))

def bleu4(ref, cand):
    precs = []
    for n in range(1, 5):
        c, r = ngrams(cand, n), ngrams(ref, n)
        clipped = sum(min(cnt, r[g]) for g, cnt in c.items())
        precs.append((clipped, sum(c.values())))
    bp = 1.0 if len(cand) > len(ref) else math.exp(1 - len(ref) / len(cand))
    geo = math.exp(sum(math.log(m / t) for m, t in precs) / 4)
    return dict(precisions=precs, bp=bp, geo=geo, bleu=bp * geo)

def rouge_n(ref, cand, n):
    r, c = ngrams(ref, n), ngrams(cand, n)
    ov = sum((r & c).values()); P = ov / sum(c.values()); R = ov / sum(r.values())
    return dict(overlap=ov, P=P, R=R, F1=2 * P * R / (P + R) if ov else 0.0)

def lcs(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a)):
        for j in range(len(b)):
            dp[i + 1][j + 1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j + 1], dp[i + 1][j])
    # back-track one LCS for display
    i, j, seq = len(a), len(b), []
    while i and j:
        if a[i - 1] == b[j - 1]:
            seq.append(a[i - 1]); i -= 1; j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return dp[-1][-1], seq[::-1]

def rouge_l(ref, cand):
    L, seq = lcs(ref, cand); P = L / len(cand); R = L / len(ref)
    return dict(lcs=L, seq=seq, P=P, R=R, F1=2 * P * R / (P + R))

def meteor_exact(ref, cand, alpha=0.9, beta=3.0, gamma=0.5):
    """METEOR (Lavie & Agarwal 2007 parameters, as used by NLTK) with exact matching.
    Alignment chosen to minimise the number of chunks."""
    m = sum((Counter(ref) & Counter(cand)).values())
    P, R = m / len(cand), m / len(ref)
    fmean = P * R / (alpha * P + (1 - alpha) * R)
    # alignment: 'the product quality is good' -> ref[5:10], 'but' -> ref[4], 'delivery was' -> ref[1:3]
    align = {0: 5, 1: 6, 2: 7, 3: 8, 4: 9, 5: 4, 6: 1, 7: 2}
    chunks, prev = 0, None
    for ci in sorted(align):
        if prev is None or ci != prev[0] + 1 or align[ci] != prev[1] + 1:
            chunks += 1
        prev = (ci, align[ci])
    penalty = gamma * (chunks / m) ** beta
    return dict(matches=m, P=P, R=R, fmean=fmean, chunks=chunks, penalty=penalty, meteor=fmean * (1 - penalty))

def perplexity(probs):
    nll = -sum(math.log(p) for p in probs) / len(probs)
    return dict(sum_log=sum(math.log(p) for p in probs), mean_nll=nll, ppl=math.exp(nll),
                geo_mean_p=math.exp(-nll))

## (1) Zero-shot accuracy with 95% confidence interval
Wald: $\hat p \pm 1.96\sqrt{\hat p(1-\hat p)/n}$; Wilson score interval shown as a more robust alternative.

In [2]:
a1 = accuracy_ci(CM1)
print(f"Accuracy = {a1['correct']}/{a1['n']} = {a1['acc']:.4f}   SE = {a1['se']:.4f}")
print(f"95% Wald CI   = [{a1['wald'][0]:.4f}, {a1['wald'][1]:.4f}]")
print(f"95% Wilson CI = [{a1['wilson'][0]:.4f}, {a1['wilson'][1]:.4f}]")

Accuracy = 230/300 = 0.7667   SE = 0.0244
95% Wald CI   = [0.7188, 0.8145]
95% Wilson CI = [0.7156, 0.8110]


## (2) Per-class precision, recall, F1 (zero-shot)

In [3]:
def print_pc(cm):
    print(f"{'class':<9}{'TP':>4}{'FP':>4}{'FN':>4}{'P':>8}{'R':>8}{'F1':>8}{'support':>9}")
    for r in per_class(cm):
        print(f"{r['cls']:<9}{r['tp']:>4}{r['fp']:>4}{r['fn']:>4}{r['P']:>8.4f}{r['R']:>8.4f}{r['F1']:>8.4f}{r['support']:>9}")
print_pc(CM1)

class      TP  FP  FN       P       R      F1  support
Positive  100  28  20  0.7812  0.8333  0.8065      120
Negative   80  15  20  0.8421  0.8000  0.8205      100
Neutral    50  27  30  0.6494  0.6250  0.6369       80


## (3) Macro, weighted and micro F1 (zero-shot)
For single-label multi-class data micro-P = micro-R = micro-F1 = accuracy.

In [4]:
def print_avg(cm):
    a = averages(cm)
    print(f"macro-F1    = {a['macro_F1']:.4f}  (macro-P {a['macro_P']:.4f}, macro-R {a['macro_R']:.4f})")
    print(f"weighted-F1 = {a['weighted_F1']:.4f}")
    print(f"micro-F1    = {a['micro_F1']:.4f}  (= accuracy)")
print_avg(CM1)

macro-F1    = 0.7546  (macro-P 0.7576, macro-R 0.7528)
weighted-F1 = 0.7659
micro-F1    = 0.7667  (= accuracy)


## (4) Stage-1 error analysis

In [5]:
off = [(CLASSES[i], CLASSES[j], CM1[i, j]) for i in range(3) for j in range(3) if i != j]
for a, p, c in sorted(off, key=lambda x: -x[2]):
    print(f"actual {a:<8} -> predicted {p:<8}: {c:>3}  ({c / CM1.sum() * 100:.1f}% of all reviews)")
neutral_errs = CM1[2, :2].sum() + CM1[:2, 2].sum()
print(f"\nErrors involving Neutral: {neutral_errs} of {CM1.sum() - np.trace(CM1)} total errors")

actual Neutral  -> predicted Positive:  20  (6.7% of all reviews)
actual Positive -> predicted Neutral :  15  (5.0% of all reviews)
actual Negative -> predicted Neutral :  12  (4.0% of all reviews)
actual Neutral  -> predicted Negative:  10  (3.3% of all reviews)
actual Negative -> predicted Positive:   8  (2.7% of all reviews)
actual Positive -> predicted Negative:   5  (1.7% of all reviews)

Errors involving Neutral: 57 of 70 total errors


## (5) Stage 2 re-evaluation and comparison

In [6]:
a2 = accuracy_ci(CM2)
print(f"Accuracy = {a2['correct']}/{a2['n']} = {a2['acc']:.4f}; Wald CI [{a2['wald'][0]:.4f}, {a2['wald'][1]:.4f}]; "
      f"Wilson CI [{a2['wilson'][0]:.4f}, {a2['wilson'][1]:.4f}]")
print_pc(CM2); print_avg(CM2)
print("\nDelta (Stage2 - Stage1):")
for r1_, r2_ in zip(per_class(CM1), per_class(CM2)):
    print(f"  {r1_['cls']:<9} dP={r2_['P']-r1_['P']:+.4f}  dR={r2_['R']-r1_['R']:+.4f}  dF1={r2_['F1']-r1_['F1']:+.4f}")
A1, A2 = averages(CM1), averages(CM2)
for k in ["macro_F1", "weighted_F1", "micro_F1"]:
    print(f"  {k:<12} {A1[k]:.4f} -> {A2[k]:.4f}  ({A2[k]-A1[k]:+.4f})")
# two-proportion z (treats stages as independent; conservative since reviews are paired)
p_pool = (a1['correct'] + a2['correct']) / 600
z = (a2['acc'] - a1['acc']) / math.sqrt(p_pool * (1 - p_pool) * (2 / 300))
print(f"two-proportion z = {z:.2f}")

Accuracy = 254/300 = 0.8467; Wald CI [0.8059, 0.8874]; Wilson CI [0.8015, 0.8830]
class      TP  FP  FN       P       R      F1  support
Positive  106  15  14  0.8760  0.8833  0.8797      120
Negative   84  11  16  0.8842  0.8400  0.8615      100
Neutral    64  20  16  0.7619  0.8000  0.7805       80
macro-F1    = 0.8406  (macro-P 0.8407, macro-R 0.8411)
weighted-F1 = 0.8472
micro-F1    = 0.8467  (= accuracy)

Delta (Stage2 - Stage1):
  Positive  dP=+0.0948  dR=+0.0500  dF1=+0.0732
  Negative  dP=+0.0421  dR=+0.0400  dF1=+0.0410
  Neutral   dP=+0.1126  dR=+0.1750  dF1=+0.1435
  macro_F1     0.7546 -> 0.8406  (+0.0859)
  weighted_F1  0.7659 -> 0.8472  (+0.0812)
  micro_F1     0.7667 -> 0.8467  (+0.0800)
two-proportion z = 2.48


## (6) Prompt stability over 5 example orderings

In [7]:
print(f"mean = {RUNS.mean():.4f}, sample SD (n-1) = {RUNS.std(ddof=1):.4f}, population SD (n) = {RUNS.std(ddof=0):.4f}")
print(f"range = {RUNS.min():.3f}-{RUNS.max():.3f}, CV = {RUNS.std(ddof=1)/RUNS.mean()*100:.2f}%")
print(f"sampling SE of a single accuracy on 300 items ~ {math.sqrt(RUNS.mean()*(1-RUNS.mean())/300):.4f}")

mean = 0.8454, sample SD (n-1) = 0.0067, population SD (n) = 0.0060
range = 0.837-0.853, CV = 0.80%
sampling SE of a single accuracy on 300 items ~ 0.0209


## (7) Justification quality: BLEU-4, ROUGE-1/2/L, METEOR

In [8]:
b = bleu4(REF, CAND)
for n, (m, t) in enumerate(b['precisions'], 1):
    print(f"p{n} = {m}/{t} = {m/t:.4f}")
print(f"BP = exp(1 - {len(REF)}/{len(CAND)}) = {b['bp']:.4f};  geometric mean = {b['geo']:.4f};  BLEU-4 = {b['bleu']:.4f}")
for n in (1, 2):
    r = rouge_n(REF, CAND, n); print(f"ROUGE-{n}: overlap {r['overlap']}, P {r['P']:.4f}, R {r['R']:.4f}, F1 {r['F1']:.4f}")
rl = rouge_l(REF, CAND); print(f"ROUGE-L: LCS {rl['lcs']} {rl['seq']}, P {rl['P']:.4f}, R {rl['R']:.4f}, F1 {rl['F1']:.4f}")
m = meteor_exact(REF, CAND)
print(f"METEOR: m={m['matches']}, P={m['P']:.4f}, R={m['R']:.4f}, Fmean={m['fmean']:.4f}, chunks={m['chunks']}, "
      f"penalty={m['penalty']:.4f}, METEOR={m['meteor']:.4f}")
try:
    from nltk.translate.bleu_score import sentence_bleu
    from nltk.translate.meteor_score import meteor_score
    print("nltk check: BLEU", round(sentence_bleu([REF], CAND), 4), " METEOR", round(meteor_score([REF], CAND), 4))
    from rouge_score import rouge_scorer
    s = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL']).score(' '.join(REF), ' '.join(CAND))
    print("rouge_score check:", {k: round(v.fmeasure, 4) for k, v in s.items()})
except Exception as e:
    print("library check skipped:", e)

p1 = 8/9 = 0.8889
p2 = 5/8 = 0.6250
p3 = 3/7 = 0.4286
p4 = 2/6 = 0.3333
BP = exp(1 - 10/9) = 0.8948;  geometric mean = 0.5308;  BLEU-4 = 0.4750
ROUGE-1: overlap 8, P 0.8889, R 0.8000, F1 0.8421
ROUGE-2: overlap 5, P 0.6250, R 0.5556, F1 0.5882
ROUGE-L: LCS 5 ['the', 'product', 'quality', 'is', 'good'], P 0.5556, R 0.5000, F1 0.5263
METEOR: m=8, P=0.8889, R=0.8000, Fmean=0.8081, chunks=3, penalty=0.0264, METEOR=0.7868
nltk check: BLEU 0.475  METEOR 0.7868
rouge_score check: {'rouge1': 0.8421, 'rouge2': 0.5882, 'rougeL': 0.5263}


## (8) Perplexity of the generated justification
$\mathrm{PPL}=\exp\left(-\frac1N\sum_i \ln p_i\right)$

In [9]:
pp = perplexity(TOKEN_PROBS)
for tok, p in zip(CAND, TOKEN_PROBS):
    print(f"{tok:<9} p={p:.2f}  -ln p={-math.log(p):.4f}")
print(f"sum ln p = {pp['sum_log']:.4f}; mean NLL = {pp['mean_nll']:.4f}; PPL = {pp['ppl']:.4f}; geometric-mean p = {pp['geo_mean_p']:.4f}")

the       p=0.60  -ln p=0.5108
product   p=0.40  -ln p=0.9163
quality   p=0.50  -ln p=0.6931
is        p=0.80  -ln p=0.2231
good      p=0.70  -ln p=0.3567
but       p=0.30  -ln p=1.2040
delivery  p=0.25  -ln p=1.3863
was       p=0.50  -ln p=0.6931
slow      p=0.20  -ln p=1.6094
sum ln p = -7.5929; mean NLL = 0.8437; PPL = 2.3249; geometric-mean p = 0.4301
